In [1]:
# Autoreload custom modules
%load_ext autoreload
%autoreload 2

In [2]:
import sys, os

import pandas as pd
import json

import numpy as np
import jax
import jax.numpy as jnp
import optax

import time
import importlib
import matplotlib.pyplot as plt

In [3]:
# import custom modules
sys.path.append('..')

from models.models import DecoderOnlyTransformer
from util.data_loader import load_data, encode, decode, get_batch
from util.optimization import train_step, loss_and_metrics, create_train_state

In [4]:
# initialize jax random key
key = jax.random.key(0)

In [5]:
# load dataset
train_data, test_data, ctoi, itoc, vocab_size = load_data(encoded_path='../data/encoded.pkl')

In [7]:
# create a basic training loop just for sanity check that the model runs
niters = 100
batch_size = 16
seq_len = 128
learning_rate = 1e-3

model, params = create_train_state(key, vocab_size, 256, 4, 4, seq_len, dropout_rate=0.2)
tx = optax.adamw(learning_rate)
opt_state = tx.init(params)


for i in range(niters):
    input, target = get_batch(train_data, batch_size, seq_len)

    params_new, opt_state_new, metrics = train_step(model, params, opt_state, input, target, tx, key=key)

    # update params and opt_state
    params = params_new
    opt_state = opt_state_new

    # Evaluate on test set every 10 iters
    if i % 10 == 0:
        B_test, T_test = 1024, 64
        test_input, test_target = get_batch(test_data, batch_size, seq_len)
        test_logits = model.apply({'params': params}, test_input, deterministic=True) # Prevent dropout during evaluation
        test_loss, test_metrics = loss_and_metrics(test_logits, test_target)
        print(f"Iter {i}, Train Loss: {metrics['loss']:.4f}, Test Loss: {test_loss:.4f}")


Iter 0, Train Loss: 3.9238, Test Loss: 5.7073
Iter 10, Train Loss: 3.0006, Test Loss: 2.8574
Iter 20, Train Loss: 2.8180, Test Loss: 2.7226
Iter 30, Train Loss: 2.6986, Test Loss: 2.6438
Iter 40, Train Loss: 2.5948, Test Loss: 2.5592
Iter 50, Train Loss: 2.5350, Test Loss: 2.5200
Iter 60, Train Loss: 2.4884, Test Loss: 2.4729
Iter 70, Train Loss: 2.4231, Test Loss: 2.4739
Iter 80, Train Loss: 2.5102, Test Loss: 2.4449


KeyboardInterrupt: 